这是ChatGpt写的代码

In [ ]:
import math
from typing import Callable

import jax
import jax.numpy as jnp
from flax import nnx
import netket as nk


# =========================
# utilities
# =========================

def default_kernel_init(key, shape, dtype):
    """Default initializer for trainable parameters."""
    return jax.nn.initializers.normal(stddev=0.02)(key, shape, dtype)


def _logdet_cmplx(A: jax.Array) -> jax.Array:
    """
    Computes log(det(A)) for a square matrix A, allowing for complex values.

    Returns
    -------
    logdet : complex scalar
        log(det(A)) = log|det(A)| + i arg(det(A))
    """
    sign, logabsdet = jnp.linalg.slogdet(A)
    return logabsdet + jnp.log(sign + 0j)


# =========================
# normalization
# =========================

class RMSNorm(nnx.Module):
    def __init__(
        self,
        dim: int,
        eps: float = 1e-6,
        param_dtype=jnp.float32,
        *,
        rngs: nnx.Rngs,
    ):
        del rngs  # no randomness needed
        self.eps = eps
        self.scale = nnx.Param(jnp.ones((dim,), dtype=param_dtype))

    def __call__(self, x: jax.Array) -> jax.Array:
        # x shape: (..., dim)
        rms = jnp.sqrt(jnp.mean(jnp.square(jnp.abs(x)), axis=-1, keepdims=True) + self.eps)
        return (x / rms) * self.scale


# =========================
# feed-forward block
# =========================

class FeedForward(nnx.Module):
    def __init__(
        self,
        dim: int,
        hidden_dim: int,
        kernel_init: Callable = default_kernel_init,
        param_dtype=jnp.float32,
        *,
        rngs: nnx.Rngs,
    ):
        self.fc1 = nnx.Linear(
            in_features=dim,
            out_features=hidden_dim,
            kernel_init=kernel_init,
            param_dtype=param_dtype,
            rngs=rngs,
        )
        self.fc2 = nnx.Linear(
            in_features=hidden_dim,
            out_features=dim,
            kernel_init=kernel_init,
            param_dtype=param_dtype,
            rngs=rngs,
        )

    def __call__(self, x: jax.Array) -> jax.Array:
        x = self.fc1(x)
        x = jax.nn.gelu(x)
        x = self.fc2(x)
        return x


# =========================
# multi-head self attention
# =========================

class MultiHeadSelfAttention(nnx.Module):
    def __init__(
        self,
        dim: int,
        n_heads: int,
        kernel_init: Callable = default_kernel_init,
        param_dtype=jnp.float32,
        *,
        rngs: nnx.Rngs,
    ):
        if dim % n_heads != 0:
            raise ValueError(f"dim={dim} must be divisible by n_heads={n_heads}")

        self.dim = dim
        self.n_heads = n_heads
        self.head_dim = dim // n_heads

        self.q_proj = nnx.Linear(
            in_features=dim,
            out_features=dim,
            kernel_init=kernel_init,
            param_dtype=param_dtype,
            rngs=rngs,
        )
        self.k_proj = nnx.Linear(
            in_features=dim,
            out_features=dim,
            kernel_init=kernel_init,
            param_dtype=param_dtype,
            rngs=rngs,
        )
        self.v_proj = nnx.Linear(
            in_features=dim,
            out_features=dim,
            kernel_init=kernel_init,
            param_dtype=param_dtype,
            rngs=rngs,
        )
        self.o_proj = nnx.Linear(
            in_features=dim,
            out_features=dim,
            kernel_init=kernel_init,
            param_dtype=param_dtype,
            rngs=rngs,
        )

    def __call__(self, x: jax.Array) -> jax.Array:
        """
        Parameters
        ----------
        x : array, shape (L, dim)
            Sequence of length L.

        Returns
        -------
        out : array, shape (L, dim)
        """
        L = x.shape[0]

        q = self.q_proj(x)  # (L, dim)
        k = self.k_proj(x)  # (L, dim)
        v = self.v_proj(x)  # (L, dim)

        # reshape to heads: (L, n_heads, head_dim) -> (n_heads, L, head_dim)
        q = q.reshape(L, self.n_heads, self.head_dim).transpose(1, 0, 2)
        k = k.reshape(L, self.n_heads, self.head_dim).transpose(1, 0, 2)
        v = v.reshape(L, self.n_heads, self.head_dim).transpose(1, 0, 2)

        # attention scores: (n_heads, L, L)
        scores = jnp.einsum("hqd,hkd->hqk", q, k) / math.sqrt(self.head_dim)
        attn = jax.nn.softmax(scores, axis=-1)

        # weighted sum: (n_heads, L, head_dim)
        out = jnp.einsum("hqk,hkd->hqd", attn, v)

        # merge heads: (L, dim)
        out = out.transpose(1, 0, 2).reshape(L, self.dim)
        out = self.o_proj(out)
        return out


# =========================
# transformer block
# =========================

class TransformerBlock(nnx.Module):
    def __init__(
        self,
        dim: int,
        n_heads: int,
        mlp_hidden_dim: int,
        kernel_init: Callable = default_kernel_init,
        param_dtype=jnp.float32,
        *,
        rngs: nnx.Rngs,
    ):
        self.norm1 = RMSNorm(dim, param_dtype=param_dtype, rngs=rngs)
        self.attn = MultiHeadSelfAttention(
            dim=dim,
            n_heads=n_heads,
            kernel_init=kernel_init,
            param_dtype=param_dtype,
            rngs=rngs,
        )
        self.norm2 = RMSNorm(dim, param_dtype=param_dtype, rngs=rngs)
        self.ff = FeedForward(
            dim=dim,
            hidden_dim=mlp_hidden_dim,
            kernel_init=kernel_init,
            param_dtype=param_dtype,
            rngs=rngs,
        )

    def __call__(self, x: jax.Array) -> jax.Array:
        x = x + self.attn(self.norm1(x))
        x = x + self.ff(self.norm2(x))
        return x


# =========================
# main ansatz
# =========================

class LogTransformerBackflow(nnx.Module):
    """
    Log-amplitude of a Slater determinant with Transformer-generated backflow orbitals.

    Wavefunction:
        log Ψ(n) = log det A(n)
        A(n) = M + F_theta(n)

    where:
        - M is a trainable bare-orbital matrix
        - F_theta(n) is produced by a Transformer acting on occupation tokens

    Notes
    -----
    1. This implementation assumes the input configuration `n` is a binary occupation vector
       of shape (..., N), where N = hilbert.size.
    2. For fermionic Hilbert spaces in NetKet, `hilbert.size` is the length of the occupation
       string. This is the correct row dimension for the determinant matrix.
    3. The determinant is built by selecting the occupied rows.
    """

    hilbert: nk.hilbert.SpinOrbitalFermions

    def __init__(
        self,
        hilbert,
        d_model: int = 64,
        n_heads: int = 4,
        n_layers: int = 2,
        mlp_hidden_dim: int = 128,
        kernel_init: Callable = default_kernel_init,
        param_dtype=jnp.float32,
        *,
        rngs: nnx.Rngs,
    ):
        self.hilbert = hilbert
        self.d_model = d_model
        self.n_heads = n_heads
        self.n_layers = n_layers
        self.mlp_hidden_dim = mlp_hidden_dim
        self.param_dtype = param_dtype

        key = rngs.params()

        # Number of single-particle orbitals in the occupation string.
        # This should match the length of the input configuration.
        self.n_modes = self.hilbert.size

        # Total number of fermions.
        # For SpinOrbitalFermions with fixed particle number, this is the number
        # of occupied modes in each configuration.
        self.n_fermions = self.hilbert.n_fermions

        # Bare orbital matrix M of shape (n_modes, n_fermions)
        self.M = nnx.Param(
            kernel_init(
                key,
                (self.n_modes, self.n_fermions),
                param_dtype,
            )
        )

        # Token embedding for binary occupations {0,1}
        # shape: (2, d_model)
        token_key = rngs.params()
        self.token_embedding = nnx.Param(
            kernel_init(token_key, (2, d_model), param_dtype)
        )

        # Learnable positional embedding of shape (n_modes, d_model)
        pos_key = rngs.params()
        self.pos_embedding = nnx.Param(
            kernel_init(pos_key, (self.n_modes, d_model), param_dtype)
        )

        # Transformer stack
        self.blocks = [
            TransformerBlock(
                dim=d_model,
                n_heads=n_heads,
                mlp_hidden_dim=mlp_hidden_dim,
                kernel_init=kernel_init,
                param_dtype=param_dtype,
                rngs=rngs,
            )
            for _ in range(n_layers)
        ]

        # Map final site features to orbital corrections
        # shape: (n_modes, d_model) -> (n_modes, n_fermions)
        self.to_orbitals = nnx.Linear(
            in_features=d_model,
            out_features=self.n_fermions,
            kernel_init=kernel_init,
            param_dtype=param_dtype,
            rngs=rngs,
        )

    def _single_logpsi(self, n: jax.Array) -> jax.Array:
        """
        Evaluate log Ψ(n) for a single configuration n of shape (n_modes,).
        """
        # Ensure integer occupation tokens in {0,1}
        n_int = n.astype(jnp.int32)

        # Token embedding lookup: (n_modes, d_model)
        x = self.token_embedding[n_int] + self.pos_embedding

        # Transformer
        for block in self.blocks:
            x = block(x)

        # Backflow correction: (n_modes, n_fermions)
        F = self.to_orbitals(x)

        # Full orbital matrix
        Phi = self.M + F

        # Occupied indices
        R = jnp.nonzero(n_int, size=self.n_fermions, fill_value=0)[0]

        # Selected square matrix: (n_fermions, n_fermions)
        A = Phi[R]

        return _logdet_cmplx(A)

    def __call__(self, n: jax.Array) -> jax.Array:
        """
        Batched evaluation.

        Parameters
        ----------
        n : array, shape (..., n_modes)

        Returns
        -------
        logpsi : array, shape (...)
        """
        if n.shape[-1] != self.n_modes:
            raise ValueError(
                f"Input last dimension must be {self.n_modes}, got {n.shape[-1]}"
            )

        if n.ndim == 1:
            return self._single_logpsi(n)

        batch_shape = n.shape[:-1]
        n_flat = n.reshape((-1, n.shape[-1]))
        out = jax.vmap(self._single_logpsi)(n_flat)
        return out.reshape(batch_shape)

最小使用示例

In [ ]:
import jax
from flax import nnx
import netket as nk

# 例子：8 个 spin-orbitals，总粒子数 4
hilbert = nk.hilbert.SpinOrbitalFermions(n_orbitals=8, n_fermions=4)

model = LogTransformerBackflow(
    hilbert=hilbert,
    d_model=64,
    n_heads=4,
    n_layers=2,
    mlp_hidden_dim=128,
    param_dtype=jnp.float32,
    rngs=nnx.Rngs(0),
)

# 单个配置：长度 = hilbert.size
n = jnp.array([1, 0, 1, 0, 1, 0, 1, 0], dtype=jnp.int32)
logpsi = model(n)
print("logpsi =", logpsi)

# batched
nb = jnp.array([
    [1, 0, 1, 0, 1, 0, 1, 0],
    [1, 1, 0, 0, 1, 0, 1, 0],
], dtype=jnp.int32)
logpsi_b = model(nb)
print("batched logpsi =", logpsi_b)

2. 这是gemini写的代码

In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx
from functools import partial

class TransformerBlock(nnx.Module):
    def __init__(self, d_model: int, num_heads: int, rngs: nnx.Rngs):
        # 多头自注意力机制
        self.attention = nnx.MultiHeadAttention(
            num_heads=num_heads,
            in_features=d_model,
            qkv_features=d_model,
            out_features=d_model,
            decode=False,
            rngs=rngs,
        )
        self.ln_1 = nnx.LayerNorm(d_model, rngs=rngs)
        self.ln_2 = nnx.LayerNorm(d_model, rngs=rngs)
        # 逐位置前馈网络 (FFN)
        self.mlp = nnx.Sequential(
            nnx.Linear(d_model, 4 * d_model, rngs=rngs),
            nnx.gelu,
            nnx.Linear(4 * d_model, d_model, rngs=rngs),
        )

    def __call__(self, x: jax.Array) -> jax.Array:
        # Pre-LN 架构（在现代 Transformer 中更易优化）
        x = x + self.attention(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

class TransformerBackflow(nnx.Module):
    def __init__(
        self,
        n_orbitals: int,
        n_fermions: int,
        d_model: int,
        num_heads: int,
        num_layers: int,
        rngs: nnx.Rngs
    ):
        self.n_orbitals = n_orbitals
        self.d_model = d_model

        # 1. 词嵌入：将占据数 (0 和 1) 映射为高维向量
        self.token_emb = nnx.Embed(num_embeddings=2, features=d_model, rngs=rngs)

        # 2. 位置编码：为每个轨道分配一个固定的几何/能量身份
        self.pos_emb = nnx.Param(
            jax.nn.initializers.normal(0.02)(rngs.params(), (n_orbitals, d_model))
        )

        # 3. Transformer 层叠 (NNX 会自动追踪列表里的模块)
        self.blocks = [
            TransformerBlock(d_model, num_heads, rngs)
            for _ in range(num_layers)
        ]

        # 4. 最终输出层：将隐藏维度投影到所需的费米子数量
        self.ln_final = nnx.LayerNorm(d_model, rngs=rngs)
        self.head = nnx.Linear(in_features=d_model, out_features=n_fermions, rngs=rngs)

    def __call__(self, n: jax.Array) -> jax.Array:
        # 输入形状: (..., n_orbitals)
        
        # 将输入强制转换为整数，以作为 Embed 层的索引
        n_int = n.astype(jnp.int32)

        # 获取嵌入向量，形状变为 (..., n_orbitals, d_model)
        x = self.token_emb(n_int)

        # 加上位置编码（利用 JAX 的自动广播机制处理 Batch 维度）
        x = x + self.pos_emb.value

        # 穿过所有 Transformer 块
        for block in self.blocks:
            x = block(x)

        # 最终投影，形状变为 (..., n_orbitals, n_fermions)
        x = self.ln_final(x)
        F = self.head(x)
        
        return F